# BTC Insider Wallet Intelligence

Explores Bitcoin insider wallet signals joined to BTC price from
`mart_signals_daily`. Requires wallet sync + dbt build.

In [1]:
from pathlib import Path

import plotly.graph_objects as go
import polars as pl
from dotenv import load_dotenv

from ccquant.forecasting import load_signals_panel, load_wallet_panel

load_dotenv()
DB = Path("data/ccquant.duckdb")

In [2]:
if DB.exists():
    signals = load_signals_panel(DB).filter(pl.col("symbol") == "BTC")
    wallet = load_wallet_panel(DB).filter(pl.col("chain") == "bitcoin")
    display(signals.tail(5))
    display(wallet.tail(5))
else:
    print("No database found — run ccquant sync wallets first")

No database found — run ccquant sync wallets first


In [3]:
if DB.exists() and not signals.is_empty():
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=signals["date"], y=signals["close"], name="BTC close"))
    if "insider_netflow_usd" in signals.columns:
        fig.add_trace(
            go.Bar(
                x=signals["date"],
                y=signals["insider_netflow_usd"],
                name="insider netflow USD",
                yaxis="y2",
            )
        )
    fig.update_layout(title="BTC price vs insider wallet netflow", yaxis2=dict(overlaying="y", side="right"))
    fig.show()